# 融合算子与 fp16 GEMM 塌陷 —— 云端受控复现

在 **Colab T4 / Kaggle P100** 上跑，目的有三个：

1. **决定性对照实验**：T4 是 `sm_75` 但**带 Tensor Core**，GTX 1650 是 `sm_75` 但**无 Tensor Core**。
   若 fp16 GEMM 的 M=1→2 塌陷在 T4 上**消失**，则「无 Tensor Core 导致」的归因成立；
   若 T4 上**同样塌陷**，则该归因被证伪，须改写结论。
2. **三方受控对照**：eager / Triton / CUDA C++ 同卡同形状，替代此前的跨卡参照。
3. **端到端**：把融合算子接进真实 Transformer block，量整层前向的实际收益。

> 依次 Run All 即可，无需改任何参数。


## 0. 环境与硬件判定


In [ ]:
import subprocess, sys, torch
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total,driver_version,compute_cap",
                      "--format=csv"],capture_output=True,text=True).stdout)
p = torch.cuda.get_device_properties(0)
name = p.name
# GTX 16 系（1650/1660）是 Turing 但移除了 Tensor Core；T4 / RTX 20 系同为 sm_75 但保留
no_tc = ("GTX 16" in name) or ("GTX 15" in name)
print(f"torch {torch.__version__} | {name} | sm_{p.major}{p.minor} | "
      f"{p.total_memory/2**30:.1f} GiB | {p.multi_processor_count} SM")
print(f"Tensor Core: {'无（GTX 16 系）' if no_tc else '有'}  ← 本次对照的关键变量")
try:
    import triton; print("triton", triton.__version__)
except Exception: print("triton 缺失")


## 1. 装 cupy（NVRTC 运行时编译 CUDA C++，免 CUDA Toolkit）


In [ ]:
import torch, subprocess, sys
cu = (torch.version.cuda or "12").split(".")[0]
pkg = "cupy-cuda13x" if cu == "13" else "cupy-cuda12x"
print("安装", pkg)
subprocess.run([sys.executable,"-m","pip","install","-q",pkg],check=False)
import cupy; print("cupy", cupy.__version__)


## 2. 写出源文件


In [ ]:
import io, os
files = {}

files['kernel.cu'] = r'''#include <cuda_fp16.h>

#define WARP 32
#define MAX_PER_THREAD 8

__device__ __forceinline__ float to_f(float v)  { return v; }
__device__ __forceinline__ float to_f(__half v) { return __half2float(v); }
template <typename T> __device__ __forceinline__ T from_f(float v);
template <> __device__ __forceinline__ float  from_f<float>(float v)  { return v; }
template <> __device__ __forceinline__ __half from_f<__half>(float v) { return __float2half(v); }

// ---- warp 内规约：shuffle，无 shared、无 __syncthreads ----
__device__ __forceinline__ float warpReduceSum(float v) {
#pragma unroll
    for (int off = WARP / 2; off > 0; off >>= 1)
        v += __shfl_down_sync(0xffffffffu, v, off);
    return v;
}

// ---- 块内规约：warp shuffle + shared 跨 warp 汇总，结果广播回全体线程 ----
__device__ __forceinline__ float blockReduceSum(float v) {
    __shared__ float partial[WARP];
    __shared__ float bcast;
    const int lane = threadIdx.x & (WARP - 1);
    const int wid  = threadIdx.x / WARP;
    const int nwarp = (blockDim.x + WARP - 1) / WARP;

    v = warpReduceSum(v);
    if (lane == 0) partial[wid] = v;
    __syncthreads();

    v = (threadIdx.x < nwarp) ? partial[threadIdx.x] : 0.0f;
    if (wid == 0) v = warpReduceSum(v);
    if (threadIdx.x == 0) bcast = v;
    __syncthreads();
    return bcast;
}

// ============================================================
// 两遍归约版：语义与 Triton 版逐条对齐
//   s = x + residual;  mean = sum(s)/N;  d = s - mean
//   var = sum(d*d)/N;  out = d * rsqrt(var+eps) * w + b
// s 缓存在寄存器，第二遍不再读显存。
// ============================================================
template <typename T>
__global__ void fused_add_ln_2pass(
    const T* __restrict__ X, const T* __restrict__ R,
    T* __restrict__ OUT,     T* __restrict__ SUM,
    const T* __restrict__ W, const T* __restrict__ B,
    const int stride_row, const int N, const float eps, const int HAS_RESIDUAL)
{
    const long long base = (long long)blockIdx.x * stride_row;
    const T* xr = X + base;
    T* outr = OUT + base;

    float s_loc[MAX_PER_THREAD];
    float sum = 0.0f;

    int i = 0;
    for (int c = threadIdx.x; c < N; c += blockDim.x, ++i) {
        float s = to_f(xr[c]);
        if (HAS_RESIDUAL) {
            s += to_f((R + base)[c]);
            (SUM + base)[c] = from_f<T>(s);   // 残差流交还调用方，避免其重算
        }
        s_loc[i] = s;
        sum += s;
    }
    const int nloc = i;

    const float mean = blockReduceSum(sum) / (float)N;

    float sq = 0.0f;
    for (int j = 0; j < nloc; ++j) { float d = s_loc[j] - mean; sq += d * d; }
    const float var  = blockReduceSum(sq) / (float)N;
    const float rstd = rsqrtf(var + eps);

    i = 0;
    for (int c = threadIdx.x; c < N; c += blockDim.x, ++i)
        outr[c] = from_f<T>((s_loc[i] - mean) * rstd * to_f(W[c]) + to_f(B[c]));
}

// ============================================================
// 单遍归约版：E[s^2]-E[s]^2，只需一次块规约（省一次 __syncthreads 往返），
// 但相消误差更大。留作精度/性能对照。
// ============================================================
template <typename T>
__global__ void fused_add_ln_1pass(
    const T* __restrict__ X, const T* __restrict__ R,
    T* __restrict__ OUT,     T* __restrict__ SUM,
    const T* __restrict__ W, const T* __restrict__ B,
    const int stride_row, const int N, const float eps, const int HAS_RESIDUAL)
{
    const long long base = (long long)blockIdx.x * stride_row;
    const T* xr = X + base;
    T* outr = OUT + base;

    float s_loc[MAX_PER_THREAD];
    float sum = 0.0f, sqs = 0.0f;

    int i = 0;
    for (int c = threadIdx.x; c < N; c += blockDim.x, ++i) {
        float s = to_f(xr[c]);
        if (HAS_RESIDUAL) {
            s += to_f((R + base)[c]);
            (SUM + base)[c] = from_f<T>(s);
        }
        s_loc[i] = s;
        sum += s; sqs += s * s;
    }

    const float mean = blockReduceSum(sum) / (float)N;
    const float var  = blockReduceSum(sqs) / (float)N - mean * mean;
    const float rstd = rsqrtf(var + eps);

    i = 0;
    for (int c = threadIdx.x; c < N; c += blockDim.x, ++i)
        outr[c] = from_f<T>((s_loc[i] - mean) * rstd * to_f(W[c]) + to_f(B[c]));
}
'''

files['fused_ln_cuda.py'] = r'''# -*- coding: utf-8 -*-
"""Fused residual-add + LayerNorm，CUDA C++ 实现（NVRTC 运行时编译）。

与 kernels/fused_layernorm.py 的 Triton 版同语义、同 API，用于逐项对照：
Triton 的 tl.sum 隐藏了行内归约，这里必须自己写 warp shuffle + shared 跨 warp 汇总。
"""
from __future__ import annotations
import os, functools
import torch, cupy as cp

_HERE = os.path.dirname(os.path.abspath(__file__))
MAX_PER_THREAD = 8
_CTYPE = {torch.float32: "float", torch.float16: "__half"}
_CPDT  = {torch.float32: cp.float32, torch.float16: cp.float16}


@functools.lru_cache(maxsize=1)
def _module():
    src = open(os.path.join(_HERE, "kernel.cu"), encoding="utf-8").read()
    names = [f"fused_add_ln_{v}<{t}>" for v in ("2pass", "1pass")
             for t in ("float", "__half")]
    return cp.RawModule(code=src, options=("--std=c++17",), name_expressions=names)


@functools.lru_cache(maxsize=8)
def _fn(variant: str, ctype: str):
    return _module().get_function(f"fused_add_ln_{variant}<{ctype}>")


def _as_cupy(t: torch.Tensor) -> cp.ndarray:
    """零拷贝把 torch CUDA 张量包成 cupy 数组（只借指针，不搬数据）。"""
    mem = cp.cuda.UnownedMemory(t.data_ptr(), t.numel() * t.element_size(), t)
    return cp.ndarray(tuple(t.shape), dtype=_CPDT[t.dtype],
                      memptr=cp.cuda.MemoryPointer(mem, 0))


def _next_pow2(n: int) -> int:
    return 1 << (n - 1).bit_length()


def _threads_for(n: int) -> int:
    """每线程约 4 个元素；不足 32 补满一个 warp，上限 1024。"""
    return max(32, min(1024, _next_pow2(max(1, (n + 3) // 4))))


def fused_add_layernorm(x, residual, weight, bias, eps=1e-5, variant="2pass"):
    """LayerNorm(x + residual)，返回 (normed, x + residual)。

    residual=None 时退化为 LayerNorm(x)，返回 (normed, x)。
    不满足 CUDA 路径条件时回落到 PyTorch，调用方无需判断。
    """
    ok = (x.is_cuda and x.dtype in _CTYPE and x.shape[-1] <= 1024
          and weight.dtype == x.dtype and bias.dtype == x.dtype)
    if ok:
        n = x.shape[-1]
        ok = (n + _threads_for(n) - 1) // _threads_for(n) <= MAX_PER_THREAD
    if not ok:
        s = x if residual is None else x + residual
        return torch.nn.functional.layer_norm(
            s, (s.shape[-1],), weight, bias, eps), s

    xc = x.contiguous()
    n = xc.shape[-1]
    flat = xc.view(-1, n)
    rows = flat.shape[0]
    out = torch.empty_like(flat)

    has_res = residual is not None
    if has_res:
        rc = residual.contiguous().view(-1, n)
        assert rc.shape == flat.shape, "residual must match x"
        total = torch.empty_like(flat)
    else:
        rc, total = flat, flat          # kernel 不会读写，占位保持签名一致

    _fn(variant, _CTYPE[x.dtype])(
        (rows,), (_threads_for(n),),
        (_as_cupy(flat), _as_cupy(rc), _as_cupy(out), _as_cupy(total),
         _as_cupy(weight.contiguous()), _as_cupy(bias.contiguous()),
         flat.stride(0), n, float(eps), int(has_res)),
    )
    shape = x.shape
    return out.view(shape), (total.view(shape) if has_res else x)
'''

files['fused_ln_triton.py'] = r'''#!/usr/bin/env python3
"""
Fused residual-add + LayerNorm, written in Triton.

**Why this fusion and not plain LayerNorm.** PyTorch's ``nn.LayerNorm`` is
already a hand-tuned fused CUDA kernel; reimplementing it in Triton is a
predictable loss. What eager PyTorch does *not* fuse is the pre-norm residual
pattern that a Transformer block repeats twice per layer::

    x = x + sublayer(norm(x))          # add is one kernel, norm is another

Each of those touches the full ``[B, S, D]`` activation. Fusing them turns four
passes over that tensor (read x, read y, write sum; read sum, write normed) into
two (read x, read y, write sum and normed), which is the whole point: LayerNorm
at these sizes is memory-bound, not compute-bound.

The kernel computes, for each row independently:

    s = x + residual
    out = (s - mean(s)) / sqrt(var(s) + eps) * weight + bias

returning both ``s`` (the new residual stream) and ``out``. Reductions accumulate
in fp32 regardless of the storage dtype, matching what ``nn.LayerNorm`` does, so
this does not spend any of the accuracy budget.

One row is one Triton program and the whole row lives in registers, which caps
``d_model`` at the largest power of two Triton will accept for a block. Every
graded shape here has ``d_model <= 1024``, and anything wider transparently falls
back to PyTorch rather than silently producing a wrong answer.

Inference only: no backward pass is defined, because the harness only ever runs
forward under ``torch.inference_mode``.
"""

from __future__ import annotations

import torch

try:
    import triton
    import triton.language as tl
    HAVE_TRITON = True
except Exception:  # pragma: no cover - triton is absent on CPU-only installs
    HAVE_TRITON = False


MAX_FUSED_WIDTH = 1024


if HAVE_TRITON:

    @triton.jit
    def _fused_add_ln_fwd(
        X, R, OUT, SUM, W, B,
        stride_row, N, eps,
        HAS_RESIDUAL: tl.constexpr,
        BLOCK: tl.constexpr,
    ):
        row = tl.program_id(0)
        X += row * stride_row
        OUT += row * stride_row
        cols = tl.arange(0, BLOCK)
        mask = cols < N

        s = tl.load(X + cols, mask=mask, other=0.0).to(tl.float32)
        if HAS_RESIDUAL:
            R += row * stride_row
            SUM += row * stride_row
            s += tl.load(R + cols, mask=mask, other=0.0).to(tl.float32)
            # The residual stream is needed by the next sublayer, so hand it back
            # rather than making the caller recompute the add.
            tl.store(SUM + cols, s, mask=mask)

        mean = tl.sum(s, axis=0) / N
        d = tl.where(mask, s - mean, 0.0)
        var = tl.sum(d * d, axis=0) / N
        rstd = 1.0 / tl.sqrt(var + eps)

        w = tl.load(W + cols, mask=mask, other=0.0).to(tl.float32)
        b = tl.load(B + cols, mask=mask, other=0.0).to(tl.float32)
        tl.store(OUT + cols, d * rstd * w + b, mask=mask)


def _next_pow2(n: int) -> int:
    return 1 << (n - 1).bit_length()


def can_fuse(x: torch.Tensor) -> bool:
    """Whether the Triton path is usable for this tensor at all."""
    return (
        HAVE_TRITON
        and x.is_cuda
        and x.shape[-1] <= MAX_FUSED_WIDTH
        and x.dtype in (torch.float16, torch.bfloat16, torch.float32)
    )


def fused_add_layernorm(x, residual, weight, bias, eps=1e-5):
    """``LayerNorm(x + residual)``, returning ``(normed, x + residual)``.

    ``residual=None`` computes a plain ``LayerNorm(x)`` and returns
    ``(normed, x)``. Falls back to PyTorch whenever the Triton path does not
    apply, so callers never have to check.
    """
    if not can_fuse(x):
        s = x if residual is None else x + residual
        return torch.nn.functional.layer_norm(
            s, (s.shape[-1],), weight, bias, eps), s

    xc = x.contiguous()
    n = xc.shape[-1]
    flat = xc.view(-1, n)
    rows = flat.shape[0]

    out = torch.empty_like(flat)
    if residual is None:
        total = flat            # unused by the kernel; keeps the signature simple
        has_res = False
    else:
        rc = residual.contiguous().view(-1, n)
        assert rc.shape == flat.shape, "residual must match x"
        total = torch.empty_like(flat)
        has_res = True

    block = _next_pow2(n)
    # 1024 lanes is where a single row stops fitting comfortably in registers;
    # below that, fewer warps keeps occupancy up on the narrow shapes.
    num_warps = 4 if block <= 512 else 8

    _fused_add_ln_fwd[(rows,)](
        flat, rc if has_res else flat, out, total if has_res else flat,
        weight.contiguous(), bias.contiguous(),
        flat.stride(0), n, eps,
        HAS_RESIDUAL=has_res,
        BLOCK=block,
        num_warps=num_warps,
    )
    shape = x.shape
    return out.view(shape), (total.view(shape) if has_res else x)


def fused_layernorm_module(norm, x, residual=None):
    """Apply an ``nn.LayerNorm`` module through the fused kernel."""
    return fused_add_layernorm(x, residual, norm.weight, norm.bias, norm.eps)
'''

files['bench3.py'] = r'''# -*- coding: utf-8 -*-
"""受控三方对照：PyTorch eager / Triton / CUDA C++，同一张 GPU、同一组形状、同一精度门。

此前 CUDA 版只能与 Kaggle T4 上记录的 Triton 结果做跨卡参照（同为 sm_75 但芯片不同），
只能得出「量级相当」。本脚本三份实现跑在同一张卡上，是真正的受控对照。

精度以 float64 为真值。计时按单次耗时自适应迭代次数，使每组测量总时长约 200 ms，
避免小形状被噪声主导。精度与计时分阶段，阶段间释放张量——4 GiB 卡上若让三份实现的
输出同时驻留，大形状会因显存压力把计时打歪。
"""
import gc, json, statistics as st, time
import torch

import fused_ln_cuda as CUDA
import fused_ln_triton as TRI

CASES = [                       # (标签, rows, D)
    ("shape 1/5/9-11", 8192,    128),
    ("shape 2",        128,     128),
    ("shape 6",        1280000, 128),
    ("shape 7",        8192,    32),
    ("shape 8",        8192,    1024),
    ("shape 13",       65536,   128),
]
IMPLS = ["eager", "triton", "cuda2", "cuda1"]


def make(rows, D, dtype):
    torch.manual_seed(0)
    return (torch.randn(rows, D, device="cuda", dtype=dtype),
            torch.randn(rows, D, device="cuda", dtype=dtype),
            torch.randn(D, device="cuda", dtype=dtype),
            torch.randn(D, device="cuda", dtype=dtype))


def call(impl, x, r, w, b):
    if impl == "eager":
        s = x + r
        return torch.nn.functional.layer_norm(s, (s.shape[-1],), w, b, 1e-5), s
    if impl == "triton":
        return TRI.fused_add_layernorm(x, r, w, b, 1e-5)
    return CUDA.fused_add_layernorm(x, r, w, b, 1e-5,
                                    variant="2pass" if impl == "cuda2" else "1pass")


def free():
    gc.collect(); torch.cuda.empty_cache()


def accuracy(rows, D, dtype):
    x, r, w, b = make(rows, D, dtype)
    k = min(rows, 4096)
    ref = torch.nn.functional.layer_norm(
        x[:k].double() + r[:k].double(), (D,), w.double(), b.double(), 1e-5)
    errs = {}
    for impl in IMPLS:
        o, _ = call(impl, x, r, w, b)
        errs[impl] = (o[:k].double() - ref).abs().max().item()
        del o; free()
    del x, r, w, b, ref; free()
    return errs


def time_one(impl, rows, D, dtype, budget_ms=200.0):
    x, r, w, b = make(rows, D, dtype)
    fn = lambda: call(impl, x, r, w, b)
    for _ in range(10): fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(5): fn()
    torch.cuda.synchronize()
    probe = (time.perf_counter() - t0) / 5 * 1e3
    iters = max(20, min(2000, int(budget_ms / max(probe, 1e-3))))
    ts = []
    for _ in range(iters):
        s, e = torch.cuda.Event(True), torch.cuda.Event(True)
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        ts.append(s.elapsed_time(e))
    del x, r, w, b, fn; free()
    return st.median(ts)


def run(dtype, tag):
    print(f"\n{'='*104}\n### dtype = {tag}\n{'='*104}")
    print(f"{'case':<16}{'rows':>9}{'D':>6}{'eager':>9}{'triton':>9}{'cuda2':>9}{'cuda1':>9}"
          f"{'TRI/eager':>11}{'CUDA/eager':>12}{'CUDA/TRI':>10}{'err_cuda2':>11}")
    out = []
    for label, rows, D in CASES:
        try:
            errs = accuracy(rows, D, dtype)
            t = {i: time_one(i, rows, D, dtype) for i in IMPLS}
            print(f"{label:<16}{rows:>9}{D:>6}"
                  f"{t['eager']:>9.4f}{t['triton']:>9.4f}{t['cuda2']:>9.4f}{t['cuda1']:>9.4f}"
                  f"{t['eager']/t['triton']:>11.3f}{t['eager']/t['cuda2']:>12.3f}"
                  f"{t['triton']/t['cuda2']:>10.3f}{errs['cuda2']:>11.2e}")
            out.append(dict(case=label, rows=rows, D=D, dtype=tag,
                            **{f"{i}_ms": t[i] for i in IMPLS},
                            tri_vs_eager=t['eager']/t['triton'],
                            cuda_vs_eager=t['eager']/t['cuda2'],
                            cuda_vs_triton=t['triton']/t['cuda2'],
                            **{f"err_{i}": errs[i] for i in IMPLS}))
        except torch.cuda.OutOfMemoryError:
            print(f"{label:<16}{rows:>9}{D:>6}   OOM (4GB)"); free()
    return out


if __name__ == "__main__":
    p = torch.cuda.get_device_properties(0)
    import triton
    print(f"GPU: {p.name} | {p.total_memory/2**30:.1f} GiB | sm_{p.major}{p.minor} | "
          f"{p.multi_processor_count} SM")
    print(f"torch {torch.__version__} | triton {triton.__version__} | 同卡受控对照")
    out = run(torch.float32, "fp32") + run(torch.float16, "fp16")
    json.dump(out, open("bench3_results.json", "w"), indent=1)
    for k, name in [("tri_vs_eager", "Triton  对 eager"),
                    ("cuda_vs_eager", "CUDA    对 eager"),
                    ("cuda_vs_triton", "CUDA    对 Triton")]:
        v = [r[k] for r in out]
        print(f"\n{name}：中位 {st.median(v):.3f}×   范围 {min(v):.3f}× – {max(v):.3f}×")
'''

files['bench_e2e.py'] = r'''# -*- coding: utf-8 -*-
"""端到端：把融合算子接进真实 Transformer block，量前向的整体收益。

孤立算子的加速比回答不了「值不值得做」——LayerNorm 只占前向的一部分，
2× 的算子加速到了端到端可能只剩几个百分点。本脚本给出那个百分比。

pre-norm 结构每层用两次「残差加 + LayerNorm」：
    s = x + residual;  normed = LN(s);  residual = s
融合版把这两步并成一个 kernel，并把新的残差流一并返回，避免调用方重算。

三份实现共用同一套权重与同一个注意力实现（F.scaled_dot_product_attention），
差别只在 norm 这一处，因此测出的差值就是融合算子的端到端贡献。
"""
import gc, json, statistics as st, time
import torch
import torch.nn as nn
import torch.nn.functional as F

import fused_ln_cuda as CUDA
try:
    import fused_ln_triton as TRI
    HAVE_TRI = True
except Exception:
    HAVE_TRI = False


class Block(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dtype):
        super().__init__()
        self.h = n_heads
        self.dk = d_model // n_heads
        self.ln1 = nn.LayerNorm(d_model, dtype=dtype)
        self.ln2 = nn.LayerNorm(d_model, dtype=dtype)
        self.qkv = nn.Linear(d_model, 3 * d_model, dtype=dtype)
        self.proj = nn.Linear(d_model, d_model, dtype=dtype)
        self.fc1 = nn.Linear(d_model, d_ff, dtype=dtype)
        self.fc2 = nn.Linear(d_ff, d_model, dtype=dtype)

    def _attn(self, x):
        B, S, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        sh = lambda t: t.view(B, S, self.h, self.dk).transpose(1, 2)
        o = F.scaled_dot_product_attention(sh(q), sh(k), sh(v))
        return self.proj(o.transpose(1, 2).reshape(B, S, D))

    def _mlp(self, x):
        return self.fc2(F.gelu(self.fc1(x)))

    def forward(self, x, residual, mode):
        """x = 上一子层输出，residual = running 残差流。返回 (新 x, 新 residual)。"""
        if mode == "eager":
            s = x + residual
            n1 = F.layer_norm(s, (s.shape[-1],), self.ln1.weight, self.ln1.bias, self.ln1.eps)
            residual = s
            a = self._attn(n1)
            s = a + residual
            n2 = F.layer_norm(s, (s.shape[-1],), self.ln2.weight, self.ln2.bias, self.ln2.eps)
            residual = s
            return self._mlp(n2), residual
        fn = TRI.fused_add_layernorm if mode == "triton" else CUDA.fused_add_layernorm
        n1, residual = fn(x, residual, self.ln1.weight, self.ln1.bias, self.ln1.eps)
        a = self._attn(n1)
        n2, residual = fn(a, residual, self.ln2.weight, self.ln2.bias, self.ln2.eps)
        return self._mlp(n2), residual


class Stack(nn.Module):
    def __init__(self, n_layers, d_model, n_heads, d_ff, dtype):
        super().__init__()
        self.blocks = nn.ModuleList(
            [Block(d_model, n_heads, d_ff, dtype) for _ in range(n_layers)])

    def forward(self, x, mode):
        residual = torch.zeros_like(x)
        for b in self.blocks:
            x, residual = b(x, residual, mode)
        return x + residual


CONFIGS = [                     # (标签, B, S, d_model, heads, d_ff, layers)
    ("小 batch 短序列", 1,  128,  512,  8, 2048, 6),
    ("中等",            8,  256,  512,  8, 2048, 6),
    ("长序列",          2, 1024,  512,  8, 2048, 6),
    ("宽模型",          4,  256,  1024, 16, 4096, 6),
]


def timeit(fn, budget_ms=300.0):
    for _ in range(5): fn()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(3): fn()
    torch.cuda.synchronize()
    probe = (time.perf_counter() - t0) / 3 * 1e3
    iters = max(10, min(500, int(budget_ms / max(probe, 1e-3))))
    ts = []
    for _ in range(iters):
        s, e = torch.cuda.Event(True), torch.cuda.Event(True)
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        ts.append(s.elapsed_time(e))
    return st.median(ts)


@torch.inference_mode()
def run(dtype, tag):
    modes = ["eager"] + (["triton"] if HAVE_TRI else []) + ["cuda"]
    print(f"\n{'='*100}\n### dtype = {tag}\n{'='*100}")
    hdr = f"{'配置':<16}{'B×S×D':>16}{'层':>4}" + "".join(f"{m:>10}" for m in modes)
    print(hdr + f"{'CUDA增益':>10}" + (f"{'TRI增益':>10}" if HAVE_TRI else "") + f"{'max|err|':>11}")
    out = []
    for label, B, S, D, H, FF, L in CONFIGS:
        torch.manual_seed(0)
        m = Stack(L, D, H, FF, dtype).cuda().eval()
        x = torch.randn(B, S, D, device="cuda", dtype=dtype)

        ref = m(x, "eager")
        errs = {}
        for mode in modes[1:]:
            o = m(x, mode)
            errs[mode] = (o.float() - ref.float()).abs().max().item()
            del o
        t = {mode: timeit(lambda mode=mode: m(x, mode)) for mode in modes}

        row = f"{label:<16}{f'{B}x{S}x{D}':>16}{L:>4}" + "".join(f"{t[mo]:>10.3f}" for mo in modes)
        row += f"{t['eager']/t['cuda']:>9.3f}x"
        if HAVE_TRI: row += f"{t['eager']/t['triton']:>9.3f}x"
        row += f"{errs['cuda']:>11.2e}"
        print(row)
        out.append(dict(case=label, B=B, S=S, D=D, layers=L, dtype=tag,
                        **{f"{mo}_ms": t[mo] for mo in modes},
                        cuda_gain=t['eager']/t['cuda'],
                        **({'triton_gain': t['eager']/t['triton']} if HAVE_TRI else {}),
                        **{f"err_{k}": v for k, v in errs.items()}))
        del m, x, ref; gc.collect(); torch.cuda.empty_cache()
    return out


if __name__ == "__main__":
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} | sm_{p.major}{p.minor} | torch {torch.__version__} | "
          f"Triton {'可用' if HAVE_TRI else '不可用'}")
    print("说明：三份实现共用同一套权重与同一注意力实现，差别只在 norm 这一处。")
    out = run(torch.float32, "fp32") + run(torch.float16, "fp16")
    json.dump(out, open("bench_e2e_results.json", "w"), indent=1)
    g = [r["cuda_gain"] for r in out]
    print(f"\nCUDA 融合算子的端到端增益：中位 {st.median(g):.3f}×  范围 {min(g):.3f}× – {max(g):.3f}×")
    print("对照：同一算子孤立测量时中位 1.428×。两者的差距就是「算子加速被整层摊薄」的幅度。")
'''

files['gemm_cliff.py'] = r'''# -*- coding: utf-8 -*-
"""脱离 vLLM 验证：sm_75 无 Tensor Core 时，fp16 GEMM 在 M=1→2 是否发生性能塌陷。

形状取自 Qwen2.5-0.5B：hidden=896, intermediate=4864。
解码时每步的核心运算就是 [M, 896] @ [896, N]，M = 并发序列数。
"""
import torch, statistics as st

H, I = 896, 4864
SHAPES = [("MLP gate/up  [M,896]@[896,4864]", H, I),
          ("MLP down     [M,4864]@[4864,896]", I, H),
          ("QKV proj     [M,896]@[896,1152]", H, 1152)]

def bench(M, K, N, dtype, iters=200):
    a = torch.randn(M, K, device="cuda", dtype=dtype)
    b = torch.randn(K, N, device="cuda", dtype=dtype)
    for _ in range(20): a @ b
    torch.cuda.synchronize()
    ts = []
    for _ in range(iters):
        s, e = torch.cuda.Event(True), torch.cuda.Event(True)
        s.record(); a @ b; e.record(); torch.cuda.synchronize()
        ts.append(s.elapsed_time(e))
    t = st.median(ts)
    # 权重矩阵是主要流量：K*N 个元素
    gbs = K * N * a.element_size() / t * 1e3 / 2**30
    return t, gbs

p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | sm_{p.major}{p.minor} | torch {torch.__version__}")
print("注：GTX 16 系为 Turing 但移除了 Tensor Core，fp16 无张量核加速。\n")

for dtype, tag in [(torch.float16, "fp16"), (torch.float32, "fp32")]:
    print(f"########## {tag} ##########")
    for name, K, N in SHAPES:
        print(f"\n{name}")
        print(f"  {'M':>4}{'耗时 ms':>11}{'权重带宽 GiB/s':>17}{'相对 M=1':>11}")
        base = None
        for M in [1, 2, 4, 8, 16, 32, 64]:
            t, gbs = bench(M, K, N, dtype)
            if base is None: base = gbs
            print(f"  {M:>4}{t:>11.4f}{gbs:>17.1f}{gbs/base:>10.2f}×")
'''

for n, c in files.items():
    io.open(n, "w", encoding="utf-8").write(c)
    print(f"  {n:<22} {len(c.splitlines()):>4} 行")



## 3. 决定性对照：fp16 GEMM 在 M=1→2 是否塌陷

**读法**：看 fp16 那几张表的 `相对 M=1` 列。

- GTX 1650（无 TC）实测：**0.04×**（塌陷 23 倍）
- 若本卡（有 TC）该列接近 **1.0×** → 归因成立
- 若同样掉到 0.0x → **归因被证伪**，须改写结论并放弃原 issue


In [ ]:
!python gemm_cliff.py


## 4. 三方受控对照（孤立算子，同卡同形状）


In [ ]:
!python bench3.py


## 5. 端到端（真实 Transformer block 前向）


In [ ]:
!python bench_e2e.py


## 6. 结果打包下载

把两个 JSON 存下来，回本地合并进 README。


In [ ]:
import json, os
for f in ["bench3_results.json","bench_e2e_results.json"]:
    if os.path.exists(f):
        d = json.load(open(f)); print(f"{f}: {len(d)} 组")
    else:
        print(f"{f}: 缺失（对应 cell 可能失败）")
try:
    from google.colab import files as gfiles
    for f in ["bench3_results.json","bench_e2e_results.json"]:
        if os.path.exists(f): gfiles.download(f)
except Exception:
    print("非 Colab 环境：Kaggle 请从右侧 Output 面板下载，或直接复制上方表格")
